# Kombinierte RBF-SVM: Index-Katalog, REIP und PCA

Dieses Notebook rekonstruiert einen kontrollierten Vergleich von Rohspektren, dem Vegetationsindex-Katalog einschließlich REIP, `PC_1`/`PC_2` sowie der Kombination beider Merkmalserweiterungen.

Die Variantenwahl erfolgt ausschließlich über Cross-Validation auf dem Trainingsanteil. Nur der CV-Gewinner wird anschließend einmal auf dem gemeinsamen Validierungsanteil bewertet.

## 1. Abhängigkeiten und Ausgangsdateien prüfen

Die gemeinsame Split-Datei und die wiederverwendbaren Komponenten müssen vorhanden sein. Das Notebook nutzt ausschließlich `awp2`-APIs für Daten, Vorverarbeitung, Tuning und Bewertung.

In [ ]:
from importlib.metadata import version

import pandas as pd
from IPython.display import display

from awp2.config import SPLIT_FILE
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

assert SPLIT_FILE.exists(), "Erzeuge die gemeinsamen Datenartefakte mit `make data`."

{
    "scikit-learn": version("scikit-learn"),
    "pandas": version("pandas"),
    "split_file": str(SPLIT_FILE),
}

## 2. Daten und Parameter rekonstruieren

Alle Varianten verwenden dieselbe kombinierte RBF-SVM, Metadaten, die numerische Monatskodierung und standardisierte Rohspektren. Unterschiede entstehen ausschließlich durch den optionalen Index- und PCA-Zweig der gemeinsamen Pipeline.

In [ ]:
EXPERIMENT_ID = "combined_svm_index_reip_pca"
TRACKING_EXPERIMENT = "vegetation-index-features"
PARAM_GRID = {
    "estimator__C": [30.0, 100.0, 300.0],
    "estimator__gamma": [0.001, 0.003, 0.01],
}
BASE_PREPROCESSING = {
    "use_meta": True,
    "use_cyclic_month": False,
    "use_spectral_standard_scale": True,
}
FEATURE_VARIANTS = {
    "Rohspektren": PreprocessingConfig(**BASE_PREPROCESSING),
    "Index-Katalog inklusive REIP": PreprocessingConfig(
        **BASE_PREPROCESSING,
        use_vegetation_indices=True,
    ),
    "PC_1 und PC_2": PreprocessingConfig(
        **BASE_PREPROCESSING,
        use_principal_components=True,
    ),
    "Index-Katalog inklusive REIP und PCA": PreprocessingConfig(
        **BASE_PREPROCESSING,
        use_vegetation_indices=True,
        use_principal_components=True,
    ),
}
VARIANT_SLUGS = {
    "Rohspektren": "raw",
    "Index-Katalog inklusive REIP": "indices_reip",
    "PC_1 und PC_2": "principal_components",
    "Index-Katalog inklusive REIP und PCA": "indices_reip_principal_components",
}

assert len(FEATURE_VARIANTS) == len(VARIANT_SLUGS) == 4

## 3. Codezellen neu erstellen

Jede Variante wird mit denselben neun Hyperparameterkombinationen nur auf den gemeinsamen Trainings-Folds optimiert. Die Rangfolge der mittleren kombinierten Balanced Accuracy bestimmt allein, welche Variante in der folgenden Zelle auf dem Holdout läuft.

In [ ]:
tuned_by_variant = {}
for variant_name, preprocessing in FEATURE_VARIANTS.items():
    variant_slug = VARIANT_SLUGS[variant_name]
    tuned_by_variant[variant_name] = tune(
        make_combined_svm(),
        TuneConfig(
            name=f"{EXPERIMENT_ID}_{variant_slug}_search",
            approach="combined",
            experiment=TRACKING_EXPERIMENT,
            description=(
                f"RBF-SVM mit {variant_name}; C und gamma werden bei ansonsten gleicher "
                "Vorverarbeitung per CV verglichen."
            ),
            preprocessing=preprocessing,
            param_grid=PARAM_GRID,
        ),
    )

tuning_summary = (
    pd.DataFrame(
        [
            {
                "Variante": variant_name,
                "CV-BAcc kombiniert": tuned.best_score,
                "Hyperparameter": tuned.best_params,
                "Tuning-Run": tuned.run_id,
            }
            for variant_name, tuned in tuned_by_variant.items()
        ]
    )
    .sort_values("CV-BAcc kombiniert", ascending=False)
    .reset_index(drop=True)
)
winner_name = tuning_summary.loc[0, "Variante"]
tuned = tuned_by_variant[winner_name]

display(tuning_summary)
{"CV-Gewinner": winner_name, "best_params": tuned.best_params, "tuning_run": tuned.run_id}

Der nächste Lauf darf erst nach der CV-Rangfolge ausgeführt werden. Er bewertet ausschließlich den per CV gewählten Kandidaten einmal auf dem gemeinsamen Validierungsanteil.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name=f"{EXPERIMENT_ID}_{VARIANT_SLUGS[winner_name]}",
        approach="combined",
        experiment=TRACKING_EXPERIMENT,
        description=(
            f"CV-Gewinner ({winner_name}) des kontrollierten Vergleichs von Index-Katalog, "
            "REIP und PCA-Komponenten."
        ),
        preprocessing=FEATURE_VARIANTS[winner_name],
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 4. Ergebnisse validieren

Die Tabelle trennt die CV-Rangfolge aller Varianten vom einzelnen Holdout-Score des CV-Gewinners. Zusätzlich werden sämtliche Metriken, ungültige Crop/Stage-Paare und die häufigsten kombinierten Verwechslungen geprüft.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = tuning_summary.copy()
comparison["Validierungs-BAcc kombiniert"] = pd.NA
comparison.loc[
    comparison["Variante"] == winner_name,
    "Validierungs-BAcc kombiniert",
] = result.metrics.bacc_combined
assert result.metrics.invalid_combinations == 0.0

top_confusions = (
    pd.DataFrame(
        {
            "Wahr": combined_label(result.y_val),
            "Vorhergesagt": combined_label(result.y_pred),
        }
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title(f"Kombinierte RBF-SVM: {winner_name}")

display(comparison)
display(result.metrics.model_dump())
display(top_confusions)
display(tuning_ax.figure)
for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

## 5. Notebook exportieren und speichern

Nach einem dokumentationswürdigen Lauf sichern die folgenden Befehle die Diagramme für die Ansatzseite und exportieren das ausgeführte Notebook optional als HTML und Python-Skript.

In [ ]:
from pathlib import Path

from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(tuning_ax.figure, f"{EXPERIMENT_ID}_tuning", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(
    plot_confusion_matrices(result.y_val, result.y_pred),
    f"{EXPERIMENT_ID}_confusion",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_class_recall(result.y_val, result.y_pred),
    f"{EXPERIMENT_ID}_recall",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_top_confusions(result.y_val, result.y_pred),
    f"{EXPERIMENT_ID}_confusions",
    MODEL_DOCS_FIGURES_DIR,
)

NOTEBOOK_PATH = Path("notebooks/04_kombiniert/11_duac1011_svm-index-reip-pca.ipynb")
assert NOTEBOOK_PATH.exists()
{
    "cv_winner": winner_name,
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "export_command": f"uv run jupyter nbconvert --to html {NOTEBOOK_PATH}",
}